# 01 — Data Pipeline Demo

Verify the data loading pipeline:
- DataLoader batch shapes
- WeightedRandomSampler behaviour
- Augmentation visualisation

In [ ]:
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from medical_mamba.data import (
    MedMNISTFolder,
    build_weighted_sampler,
    get_train_transforms,
    get_val_transforms,
)

## 1. Load a Dataset

In [ ]:
DS_NAME = "pathmnist"
ROOT = "../data"

train_transform = get_train_transforms(DS_NAME, img_size=224)
val_transform = get_val_transforms(DS_NAME, img_size=224)

train_ds = MedMNISTFolder(DS_NAME, ROOT, "train", train_transform)
val_ds = MedMNISTFolder(DS_NAME, ROOT, "val", val_transform)

print(f"Train: {len(train_ds)} samples | Val: {len(val_ds)} samples")

## 2. Sampler Verification

In [ ]:
sampler = build_weighted_sampler(train_ds)
loader = DataLoader(train_ds, batch_size=64, sampler=sampler)

batch = next(iter(loader))
images, labels = batch["image"], batch["label"]
print(f"Batch shape: images={images.shape}, labels={labels.shape}")
print(f"Label distribution in batch: {torch.bincount(labels)}")

## 3. Augmentation Visualisation

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(20, 5))
fig.suptitle("Augmented Training Samples", fontsize=14)

for i, ax in enumerate(axes.flat):
    sample = train_ds[i]
    img, label = sample["image"], sample["label"]
    # De-normalise for display
    img_np = img.permute(1, 2, 0).numpy()
    img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
    ax.imshow(img_np)
    ax.set_title(f"Class {label.item()}", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()